# Phase 5 — Feature Engineering + LightGBM Ranker

Generates synthetic training data across **multiple origin-destination pairs** and **multiple disruption severity levels**, engineers route-level features, derives relevance labels from a utility function, and trains a LightGBM LambdaRank model.

**Why synthetic labels:** we have no real historical voyage data. Synthetic labels are derived by computing a utility score per candidate route (distance + disruption exposure, matching the paper's utility function) and ranking candidates within each query group — this gives LightGBM's LambdaRank objective real pairwise preference signal to learn from, even without ground-truth voyages.

In [63]:
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import lightgbm as lgb
from itertools import islice
import random

random.seed(42)
np.random.seed(42)

PROCESSED_DIR = "../data/processed"

# Load the base graph (has disruption_severity on nodes, base_weight on edges from Phase 3)
with open(f"{PROCESSED_DIR}/maritime_graph_disrupted.gpickle", "rb") as f:
    G = pickle.load(f)

print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph loaded: 1543 nodes, 9462 edges


In [64]:
def port_lookup_by_name(G, name_substring):
    return [
        (node_id, attrs['port_name'], attrs['country'])
        for node_id, attrs in G.nodes(data=True)
        if name_substring.lower() in attrs['port_name'].lower()
    ]

def k_shortest_paths(G, origin_id, dest_id, k=5, weight_key='disrupted_weight'):
    if origin_id not in G or dest_id not in G or not nx.has_path(G, origin_id, dest_id):
        return []
    try:
        paths_generator = nx.shortest_simple_paths(G, origin_id, dest_id, weight=weight_key)
        return list(islice(paths_generator, k))
    except Exception:
        return []

## Step 1 — Select diverse origin-destination pairs
Picks major, well-known ports across different regions so training data covers varied route geometries (not just Europe-Asia).

In [65]:
CANDIDATE_PORT_NAMES = [
    "rotterdam", "hamburg", "antwerp",           # Europe
    "singapore", "shanghai", "hong kong", "tokyo", "busan",  # Asia
    "los angeles", "long beach", "new york",      # N. America
    "santos", "buenos aires",                      # S. America
    "durban", "lagos",                              # Africa
    "sydney", "melbourne",                          # Oceania
    "dubai", "mumbai",                              # Middle East / South Asia
]

resolved_ports = {}
for name in CANDIDATE_PORT_NAMES:
    matches = port_lookup_by_name(G, name)
    if matches:
        resolved_ports[name] = matches[0][0]  # take first match's port_id
        print(f"  {name} -> {matches[0][1]} ({matches[0][2]}) [id={matches[0][0]}]")
    else:
        print(f"  {name} -> NOT FOUND, skipping")

print(f"\nResolved {len(resolved_ports)} / {len(CANDIDATE_PORT_NAMES)} ports")

  rotterdam -> Rotterdam (Netherlands) [id=31140]
  hamburg -> Hamburg (Germany) [id=30780]
  antwerp -> Antwerpen (Belgium) [id=31250]
  singapore -> Keppel - (East Singapore) (Singapore) [id=50000]
  shanghai -> Shanghai (China) [id=59970]
  hong kong -> Hong Kong (Hong Kong) [id=57840]
  tokyo -> Tokyo Ko (Japan) [id=61380]
  busan -> Busan (South Korea) [id=60390]
  los angeles -> Los Angeles (United States) [id=16080]
  long beach -> Long Beach (United States) [id=16070]
  new york -> New York City (United States) [id=7640]
  santos -> Santos (Brazil) [id=12970]
  buenos aires -> Buenos Aires (Argentina) [id=13760]
  durban -> Durban (South Africa) [id=46850]
  lagos -> Lagos (Nigeria) [id=46130]
  sydney -> North Sydney (Canada) [id=5990]
  melbourne -> Melbourne (Australia) [id=54030]
  dubai -> NOT FOUND, skipping
  mumbai -> Mumbai (Bombay) (India) [id=48840]

Resolved 18 / 19 ports


In [66]:
# Force-include ALL European-origin pairs to Asia/Middle East/Africa destinations —
# these are the pairs geographically plausible to route through the Red Sea/Suez corridor,
# so they're the ones where disruption-awareness actually matters.
EUROPE_PORTS = ['rotterdam', 'hamburg', 'antwerp']
ASIA_ME_AFRICA_PORTS = ['singapore', 'shanghai', 'hong kong', 'tokyo', 'busan', 'mumbai', 'durban', 'lagos']

FORCED_PAIRS = []
for eu in EUROPE_PORTS:
    for dest in ASIA_ME_AFRICA_PORTS:
        if eu in resolved_ports and dest in resolved_ports:
            FORCED_PAIRS.append((resolved_ports[eu], resolved_ports[dest]))

print(f"Forced Red-Sea-relevant pairs: {len(FORCED_PAIRS)}")

port_ids = list(resolved_ports.values())
all_pairs = [(a, b) for i, a in enumerate(port_ids) for b in port_ids[i+1:]]
random.shuffle(all_pairs)

MAX_OD_PAIRS = 60  # raised from 40 to accommodate more forced pairs + still have variety
od_pairs = list(FORCED_PAIRS)
for a, b in all_pairs:
    if (a, b) in od_pairs or (b, a) in od_pairs:
        continue
    if nx.has_path(G, a, b):
        od_pairs.append((a, b))
    if len(od_pairs) >= MAX_OD_PAIRS:
        break

print(f"Using {len(od_pairs)} origin-destination pairs (including {len(FORCED_PAIRS)} forced)")

Forced Red-Sea-relevant pairs: 24
Using 60 origin-destination pairs (including 24 forced)


## Step 2 — Multiple disruption severity scenarios
Reuses the disruption-tagged ports from Phase 3 (`disruption_severity` node attribute) but scales it by a scenario multiplier to simulate different crisis intensities — from no disruption to full severity — without needing new GDELT pulls.

In [67]:
SEVERITY_SCENARIOS = {
    'none': 0.0,
    'mild': 0.3,
    'moderate': 0.6,
    'severe': 1.0,
}

PENALTY_MULTIPLIER = 30.0  # matches final Phase 3 calibration

def compute_scenario_weight(G, u, v, scenario_multiplier):
    """Recomputes edge weight for a given scenario intensity without mutating the base graph."""
    base = G[u][v]['base_weight']
    sev_u = G.nodes[u].get('disruption_severity', 0.0)
    sev_v = G.nodes[v].get('disruption_severity', 0.0)
    max_sev = max(sev_u, sev_v) * scenario_multiplier
    penalty_factor = 1.0 + PENALTY_MULTIPLIER * max_sev
    return base * penalty_factor, max_sev

def build_scenario_graph(G, scenario_multiplier):
    """Returns a shallow copy of G with edge weights recomputed for this scenario."""
    G_scenario = G.copy()
    for u, v in G_scenario.edges():
        w, sev = compute_scenario_weight(G, u, v, scenario_multiplier)
        G_scenario[u][v]['scenario_weight'] = w
        G_scenario[u][v]['scenario_severity'] = sev
    return G_scenario

# Pre-build one graph per scenario (avoids recomputing per OD pair)
scenario_graphs = {
    name: build_scenario_graph(G, mult) for name, mult in SEVERITY_SCENARIOS.items()
}
print("Scenario graphs built:", list(scenario_graphs.keys()))

Scenario graphs built: ['none', 'mild', 'moderate', 'severe']


## Step 3 — Generate candidate routes and engineer features
For each (OD pair, scenario), generates K=5 candidate routes and computes route-level features.

In [68]:
HARBOR_SIZE_SCORE = {'Large': 3, 'Medium': 2, 'Small': 1, 'Very Small': 0, 'Unknown': 0}

def compute_route_features(G_scenario, path):
    """Computes a feature dict for a single candidate route."""
    total_distance = 0.0
    total_scenario_weight = 0.0
    max_exposure = 0.0
    sum_exposure = 0.0

    for i in range(len(path) - 1):
        u, v = path[i], path[i+1]
        edge = G_scenario[u][v]
        total_distance += edge['distance_nm']
        total_scenario_weight += edge['scenario_weight']
        sev = edge.get('scenario_severity', 0.0)
        max_exposure = max(max_exposure, sev)
        sum_exposure += sev

    depths = [G_scenario.nodes[p].get('cargo_pier_depth_m', np.nan) for p in path]
    depths = [d for d in depths if not np.isnan(d)]
    bottleneck_depth = min(depths) if depths else 0.0

    harbor_scores = [HARBOR_SIZE_SCORE.get(G_scenario.nodes[p].get('harbor_size', 'Unknown'), 0) for p in path]
    avg_harbor_score = float(np.mean(harbor_scores)) if harbor_scores else 0.0

    num_disrupted_ports = sum(
        1 for p in path if G_scenario.nodes[p].get('disruption_severity', 0.0) > 0
    )

    return {
        'total_distance_nm': total_distance,
        'num_hops': len(path) - 1,
        'disruption_exposure_max': max_exposure,
        'disruption_exposure_sum': sum_exposure,
        'bottleneck_depth_m': bottleneck_depth,
        'avg_harbor_score': avg_harbor_score,
        'num_disrupted_ports': num_disrupted_ports,
        'scenario_weight': total_scenario_weight,
    }

In [69]:
K_CANDIDATES = 4  # reduced per strategy since we're now pulling from 2 strategies

rows = []
query_id = 0

for origin_id, dest_id in od_pairs:
    for scenario_name, G_scenario in scenario_graphs.items():
        # Strategy A: disruption-averse candidates
        paths_safe = k_shortest_paths(G_scenario, origin_id, dest_id, k=K_CANDIDATES, weight_key='scenario_weight')
        # Strategy B: pure-distance candidates (may cut through disrupted zones)
        paths_short = k_shortest_paths(G_scenario, origin_id, dest_id, k=K_CANDIDATES, weight_key='base_weight')

        # Merge and dedupe by path tuple
        seen_paths = set()
        combined_paths = []
        for path in paths_safe + paths_short:
            path_key = tuple(path)
            if path_key not in seen_paths:
                seen_paths.add(path_key)
                combined_paths.append(path)

        if len(combined_paths) < 2:
            continue

        candidates = []
        for path in combined_paths:
            feats = compute_route_features(G_scenario, path)
            feats['path'] = path
            candidates.append(feats)

        for c in candidates:
            c['origin_id'] = origin_id
            c['dest_id'] = dest_id
            c['scenario'] = scenario_name
            c['query_id'] = query_id
            rows.append(c)

        query_id += 1

route_df = pd.DataFrame(rows)
print(f"Generated {len(route_df)} candidate routes across {query_id} queries")

variance_check = route_df.groupby('query_id')['disruption_exposure_sum'].std()
queries_with_signal = (variance_check > 0).sum()
print(f"Queries where disruption exposure varies across candidates: {queries_with_signal} / {route_df['query_id'].nunique()}")

KeyboardInterrupt: 

diagnositic

In [ ]:
variance_check = route_df.groupby('query_id')['disruption_exposure_sum'].std()
queries_with_signal = (variance_check > 0).sum()
print(f"Queries where disruption exposure varies across candidates: {queries_with_signal} / {route_df['query_id'].nunique()}")

Queries where disruption exposure varies across candidates: 84 / 240


## Step 4 — Derive synthetic relevance labels
Computes a utility score per candidate (lower = better), combining normalized distance and disruption exposure — matching the paper's utility function structure (time + risk components; cost/emissions omitted here since we lack those data sources). Within each query group, candidates are ranked by utility and assigned a relevance grade (0–4, LightGBM LambdaRank's expected label format).

In [ ]:
UTILITY_WEIGHTS = {'distance': 0.35, 'disruption': 0.65}  # was 0.6/0.4 — now prioritizes avoidance

def assign_relevance_labels(df):
    df = df.copy()
    df['relevance'] = 0

    for qid, group in df.groupby('query_id'):
        idx = group.index

        # Normalize within the group (min-max) so distance and exposure are comparable
        dist_norm = (group['total_distance_nm'] - group['total_distance_nm'].min()) / \
                    (group['total_distance_nm'].max() - group['total_distance_nm'].min() + 1e-9)
        exposure_norm = (group['disruption_exposure_sum'] - group['disruption_exposure_sum'].min()) / \
                        (group['disruption_exposure_sum'].max() - group['disruption_exposure_sum'].min() + 1e-9)

        utility = UTILITY_WEIGHTS['distance'] * dist_norm + UTILITY_WEIGHTS['disruption'] * exposure_norm

        # Lower utility = better route. Rank ascending, map to relevance grades 4 (best) down to 0 (worst)
        ranks = utility.rank(method='first', ascending=True)  # 1 = best (lowest utility)
        n = len(group)
        # Map rank 1..n to relevance (n-1)..0, capped at 4 grades
        relevance = ((n - ranks) / max(n - 1, 1) * 4).round().astype(int)
        df.loc[idx, 'relevance'] = relevance.values

    return df

route_df = assign_relevance_labels(route_df)
print(route_df[['query_id', 'total_distance_nm', 'disruption_exposure_sum', 'relevance']].head(15))
print("\nRelevance label distribution:")
print(route_df['relevance'].value_counts().sort_index())

    query_id  total_distance_nm  disruption_exposure_sum  relevance
0          0        8569.802348                    0.000          4
1          0        8569.807324                    0.000          3
2          0        8569.807481                    0.000          1
3          0        8569.820352                    0.000          0
4          1       14322.789811                    0.000          4
5          1       14322.807815                    0.000          3
6          1       14322.813308                    0.000          3
7          1       14322.814609                    0.000          2
8          1        8569.802348                    1.299          2
9          1        8569.807324                    1.299          1
10         1        8569.807481                    1.299          1
11         1        8569.820352                    1.299          0
12         2       14322.789811                    0.000          4
13         2       14322.807815                 

## Step 5 — Train/validation split (by query, not by row — prevents leakage)

In [ ]:
unique_queries = route_df['query_id'].unique()
np.random.shuffle(unique_queries)

split_idx = int(len(unique_queries) * 0.8)
train_queries = set(unique_queries[:split_idx])
val_queries = set(unique_queries[split_idx:])

train_df = route_df[route_df['query_id'].isin(train_queries)].sort_values('query_id').reset_index(drop=True)
val_df = route_df[route_df['query_id'].isin(val_queries)].sort_values('query_id').reset_index(drop=True)

print(f"Train: {len(train_df)} rows / {len(train_queries)} queries")
print(f"Val:   {len(val_df)} rows / {len(val_queries)} queries")

Train: 1052 rows / 192 queries
Val:   244 rows / 48 queries


In [ ]:
FEATURE_COLS = [
    'total_distance_nm', 'num_hops', 'disruption_exposure_max',
    'disruption_exposure_sum', 'bottleneck_depth_m', 'avg_harbor_score',
    'num_disrupted_ports'
]

def build_group_sizes(df):
    """LightGBM ranker needs group sizes = number of candidates per query, in query order."""
    return df.groupby('query_id', sort=False).size().tolist()

X_train = train_df[FEATURE_COLS]
y_train = train_df['relevance']
group_train = build_group_sizes(train_df)

X_val = val_df[FEATURE_COLS]
y_val = val_df['relevance']
group_val = build_group_sizes(val_df)

print(f"Feature matrix: {X_train.shape}, groups: {len(group_train)}, sum(group_sizes)={sum(group_train)}")

Feature matrix: (1052, 7), groups: 192, sum(group_sizes)=1052


## Step 6 — Train LightGBM LambdaRank

In [ ]:
train_data = lgb.Dataset(X_train, label=y_train, group=group_train)
val_data = lgb.Dataset(X_val, label=y_val, group=group_val, reference=train_data)

params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [1, 3, 5],
    'learning_rate': 0.05,
    'num_leaves': 15,
    'min_data_in_leaf': 5,
    'verbose': -1,
}

model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    num_boost_round=200,
    callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(period=20)]
)

Training until validation scores don't improve for 20 rounds
[20]	train's ndcg@1: 0.983681	train's ndcg@3: 0.977945	train's ndcg@5: 0.988602	val's ndcg@1: 0.906944	val's ndcg@3: 0.940734	val's ndcg@5: 0.958428
Early stopping, best iteration is:
[1]	train's ndcg@1: 1	train's ndcg@3: 0.992423	train's ndcg@5: 0.994113	val's ndcg@1: 1	val's ndcg@3: 0.994547	val's ndcg@5: 0.994997


## Step 7 — Evaluate: LightGBM ranker vs Baseline 1 (pure distance ranking)
Compares NDCG when ranking by the trained model's predicted scores vs. simply ranking by `total_distance_nm` ascending (Baseline 1's logic). If the ML ranker doesn't beat this, it isn't adding value.

In [ ]:
from sklearn.metrics import ndcg_score

def evaluate_ndcg(df, score_col, k_values=[1, 3, 5]):
    results = {k: [] for k in k_values}
    for qid, group in df.groupby('query_id'):
        true_relevance = group['relevance'].values.reshape(1, -1)
        scores = group[score_col].values.reshape(1, -1)
        for k in k_values:
            if len(group) >= 2:  # ndcg_score needs at least 2 items
                results[k].append(ndcg_score(true_relevance, scores, k=min(k, len(group))))
    return {k: np.mean(v) for k, v in results.items()}

val_df = val_df.copy()
val_df['lgbm_score'] = model.predict(X_val)
val_df['baseline_score'] = -val_df['scenario_weight']  # was -total_distance_nm — now matches what Baseline 1 actually optimizes

lgbm_ndcg = evaluate_ndcg(val_df, 'lgbm_score')

baseline_ndcg = evaluate_ndcg(val_df, 'baseline_score')

print("LightGBM Ranker NDCG:", lgbm_ndcg)
print("Baseline 1 (distance-only) NDCG:", baseline_ndcg)

comparison = pd.DataFrame({
    'LightGBM Ranker': lgbm_ndcg,
    'Baseline 1 (distance-only)': baseline_ndcg
})
comparison

LightGBM Ranker NDCG: {1: np.float64(0.6814236111111112), 3: np.float64(0.7817523595286501), 5: np.float64(0.8675138569867146)}
Baseline 1 (distance-only) NDCG: {1: np.float64(1.0), 3: np.float64(0.9999999999999999), 5: np.float64(0.9999999999999999)}


,LightGBM Ranker,Baseline 1 (distance-only)
1,0.681424,1.0
3,0.781752,1.0
5,0.867514,1.0


## Step 8 — Feature importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print(importance_df)

                   feature  importance
2  disruption_exposure_max  114.472000
5         avg_harbor_score   46.014919
0        total_distance_nm   23.109640
1                 num_hops    7.706610
4       bottleneck_depth_m    6.301450
3  disruption_exposure_sum    0.000000
6      num_disrupted_ports    0.000000


## Step 9 — Save model and training data

In [ ]:
import os
MODELS_DIR = "../data/processed/models"
os.makedirs(MODELS_DIR, exist_ok=True)

model.save_model(f"{MODELS_DIR}/lgbm_ranker.txt")
route_df.to_csv(f"{PROCESSED_DIR}/synthetic_route_training_data.csv", index=False)

print(f"Model saved to {MODELS_DIR}/lgbm_ranker.txt")
print(f"Training data saved to {PROCESSED_DIR}/synthetic_route_training_data.csv")
print(f"\nFeature columns used (save this list — needed for API inference in Phase 6):")
print(FEATURE_COLS)

Model saved to ../data/processed/models/lgbm_ranker.txt
Training data saved to ../data/processed/synthetic_route_training_data.csv

Feature columns used (save this list — needed for API inference in Phase 6):
['total_distance_nm', 'num_hops', 'disruption_exposure_max', 'disruption_exposure_sum', 'bottleneck_depth_m', 'avg_harbor_score', 'num_disrupted_ports']
